# LeetCode 3525 — Find X Value of Array II bằng Python

- Link: https://leetcode.com/problems/find-x-value-of-array-ii/
- Độ khó: **Hard**
- Chủ đề: **Cây phân đoạn** lưu "phân bố số dư của tích tiền tố", **phép gộp không giao hoán**, cập nhật điểm, chia khối

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def resultArray(self, nums: List[int], k: int, queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Dịch đề: sau khi bỏ tiền tố `nums[0..start−1]` và bỏ một hậu tố (còn lại khác rỗng), phần còn lại là một **tiền tố khác rỗng của `nums[start..]`** ⇒ đếm số tiền tố có tích `≡ x (mod k)`.
2. Thiết kế "thông tin đoạn" `(P, cnt[0..k−1])`: `P` = tích cả đoạn mod `k`, `cnt[r]` = số tiền tố khác rỗng của đoạn có tích `≡ r`.
3. Viết phép **gộp** hai đoạn kề nhau: `cnt[r] = cntA[r] + Σ_{s: PA·s ≡ r} cntB[s]` — chú ý **không giao hoán** (thứ tự trái/phải quan trọng).
4. Dùng cây phân đoạn: cập nhật điểm `O(k log n)`, truy vấn đoạn `O(k log n)`.
5. So sánh với chia khối và vét cạn.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums`, số `k` và các truy vấn `[index, value, start, x]`. Với mỗi truy vấn:

1. Gán `nums[index] = value` (**giữ lại** cho các truy vấn sau).
2. Bỏ tiền tố `nums[0..start−1]` (chỉ cho truy vấn này).
3. **x-value** = số cách bỏ một hậu tố (có thể rỗng) sao cho phần còn lại **khác rỗng** và tích các phần tử còn lại chia `k` dư `x`.

**Ví dụ 1:** `nums = [1,2,3,4,5], k = 3, queries = [[2,2,0,2],[3,3,3,0],[0,1,0,1]]` → `[2,2,2]`.

**Ví dụ 2:** `nums = [1,2,4,8,16,32], k = 4, queries = [[0,2,0,2],[0,2,0,1]]` → `[1,0]`.

**Ví dụ 3:** `nums = [1,1,2,1,1], k = 2, queries = [[2,1,0,1]]` → `[5]`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ k ≤ 5`, `1 ≤ q ≤ 2·10^4`, `1 ≤ nums[i], value ≤ 10^9`.

In [1]:
from typing import List
examples = [
    ([1, 2, 3, 4, 5], 3, [[2, 2, 0, 2], [3, 3, 3, 0], [0, 1, 0, 1]], [2, 2, 2]),
    ([1, 2, 4, 8, 16, 32], 4, [[0, 2, 0, 2], [0, 2, 0, 1]], [1, 0]),
    ([1, 1, 2, 1, 1], 2, [[2, 1, 0, 1]], [5]),
]
for nums, k, qs, exp in examples:
    print(nums, k, qs, '-> kỳ vọng', exp)

[1, 2, 3, 4, 5] 3 [[2, 2, 0, 2], [3, 3, 3, 0], [0, 1, 0, 1]] -> kỳ vọng [2, 2, 2]
[1, 2, 4, 8, 16, 32] 4 [[0, 2, 0, 2], [0, 2, 0, 1]] -> kỳ vọng [1, 0]
[1, 1, 2, 1, 1] 2 [[2, 1, 0, 1]] -> kỳ vọng [5]


## 3. Kiến thức nền

### 3.1. Tích modulo dồn

`(a·b) mod k = ((a mod k)·(b mod k)) mod k` ⇒ chỉ cần giữ các số dư nhỏ, tránh số khổng lồ.

In [2]:
k = 3
p, prefix = 1, []
for x in [1, 2, 2, 4, 5]:
    p = p * x % k
    prefix.append(p)
print('số dư của các tích tiền tố:', prefix)

số dư của các tích tiền tố: [1, 2, 1, 1, 2]


### 3.2. Bỏ hậu tố = giữ tiền tố

Mảng `a` độ dài `m`: bỏ hậu tố độ dài `0..m−1` (còn lại khác rỗng) ⇔ giữ tiền tố độ dài `m..1` ⇒ có đúng `m` cách, tương ứng `m` tiền tố khác rỗng.

### 3.3. Thông tin đoạn và phép gộp

Với đoạn `S`: `P(S)` = tích mod `k`; `cnt_S[r]` = số tiền tố khác rỗng của `S` có tích ≡ `r`. Với hai đoạn kề `A` rồi `B`:

- Tiền tố của `A·B` hoặc nằm trong `A` (đóng góp `cnt_A`), hoặc = toàn bộ `A` + một tiền tố của `B` (tích = `P(A) · tích tiền tố của B`).
- `P(A·B) = P(A)·P(B)`.

In [3]:
def leaf(v, k):
    c = [0] * k
    c[v % k] += 1
    return (v % k, c)

def combine(A, B, k):
    pa, ca = A
    pb, cb = B
    c = ca[:]
    for s in range(k):
        if cb[s]:
            c[pa * s % k] += cb[s]
    return (pa * pb % k, c)

IDENT = lambda k: (1 % k, [0] * k)      # đoạn rỗng

k = 3
seg = IDENT(k)
for x in [1, 2, 2, 4, 5]:
    seg = combine(seg, leaf(x, k), k)
print('P =', seg[0], '| cnt =', seg[1])

P = 2 | cnt = [0, 3, 2]


### 3.4. Cây phân đoạn không giao hoán

Phép gộp có tính **kết hợp** nhưng **không giao hoán** ⇒ khi truy vấn đoạn bằng cây phân đoạn lặp, phải gom **hai phía riêng**: kết quả bên trái gộp theo thứ tự trái→phải, kết quả bên phải gộp theo thứ tự phải→trái, cuối cùng `combine(trái, phải)`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Truy vấn 0 ở ví dụ 1: sau cập nhật `nums = [1,2,2,4,5]`, `start = 0`, `x = 2`. Tích tiền tố mod 3: `1, 2, 4≡1, 16≡1, 80≡2` ⇒ hai tiền tố (`[1,2]` và cả mảng) có dư 2 ⇒ đáp án 2.

Truy vấn 1: `nums = [1,2,2,3,5]`, `start = 3` ⇒ xét `[3, 5]`: tích tiền tố `0, 0` ⇒ `x = 0` có 2 cách.

⇒ Mỗi truy vấn cần **phân bố số dư của các tích tiền tố của một hậu tố mảng**, với cập nhật điểm xen giữa ⇒ cây phân đoạn.

In [4]:
for arr, x in [([1, 2, 2, 4, 5], 2), ([3, 5], 0)]:
    p, c = 1, [0] * 3
    for v in arr:
        p = p * v % 3
        c[p] += 1
    print(arr, '-> phân bố dư tiền tố', c, '-> đáp án x =', x, ':', c[x])

[1, 2, 2, 4, 5] -> phân bố dư tiền tố [0, 3, 2] -> đáp án x = 2 : 2
[3, 5] -> phân bố dư tiền tố [2, 0, 0] -> đáp án x = 0 : 2


## 5. Cách 1 — Brute Force: tính lại mỗi truy vấn

In [5]:
def x_value_brute(nums, k, queries):
    a = nums[:]
    res = []
    for idx, val, start, x in queries:
        a[idx] = val
        p, c = 1 % k, 0
        for v in a[start:]:
            p = p * v % k
            c += p == x
        res.append(c)
    return res

for nums, k, qs, exp in examples:
    print(x_value_brute(nums, k, qs), '| kỳ vọng', exp)

[2, 2, 2] | kỳ vọng [2, 2, 2]
[1, 0] | kỳ vọng [1, 0]
[5] | kỳ vọng [5]


**Độ phức tạp:** `O(n)` mỗi truy vấn ⇒ `O(n·q) = 2·10^9` ở giới hạn — quá chậm.

**Nút thắt:** mỗi truy vấn duyệt lại cả hậu tố dù chỉ một phần tử thay đổi.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề:** thông tin `(P, cnt)` của đoạn ghép tính được từ thông tin hai nửa (mục 3.3) trong `O(k)` ⇒ dùng được cây phân đoạn:

- Lá `i`: `(v mod k, e_{v mod k})` (vector đơn vị).
- Nút trong: `combine(trái, phải)`.
- Cập nhật điểm: sửa lá, tính lại các tổ tiên `O(k log n)`.
- Truy vấn `[start, n)`: gộp `O(log n)` nút theo đúng thứ tự ⇒ đọc `cnt[x]`.

Quy tắc ngắn:

- **Nếu** tiền tố nằm hẳn trong nửa trái **thì** đếm trong `cnt_trái`.
- **Nếu** tiền tố vượt sang nửa phải **thì** số dư = `P_trái · (dư tiền tố của nửa phải)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Chia khối `√n`

Mỗi khối lưu `(P, cnt)`. Cập nhật: tính lại khối chứa phần tử `O(B)`. Truy vấn: phần đầu khối chứa `start` tính thủ công, các khối sau gộp `O(k)` mỗi khối.

In [6]:
def x_value_blocks(nums, k, queries):
    a = nums[:]
    n = len(a)
    B = max(1, int(n ** 0.5))
    nb_ = (n + B - 1) // B

    def build_block(b):
        seg = IDENT(k)
        for v in a[b * B:min(n, (b + 1) * B)]:
            seg = combine(seg, leaf(v, k), k)
        return seg

    blocks = [build_block(b) for b in range(nb_)]
    res = []
    for idx, val, start, x in queries:
        a[idx] = val
        blocks[idx // B] = build_block(idx // B)
        b0 = start // B
        seg = IDENT(k)
        for v in a[start:min(n, (b0 + 1) * B)]:
            seg = combine(seg, leaf(v, k), k)
        for b in range(b0 + 1, nb_):
            seg = combine(seg, blocks[b], k)
        res.append(seg[1][x])
    return res

for nums, k, qs, exp in examples:
    print(x_value_blocks(nums, k, qs), '| kỳ vọng', exp)

[2, 2, 2] | kỳ vọng [2, 2, 2]
[1, 0] | kỳ vọng [1, 0]
[5] | kỳ vọng [5]


**Độ phức tạp:** `O((B + n/B)·k)` mỗi truy vấn ⇒ `O(k·√n)`.

### 7.2. Cách 3 — Cây phân đoạn đệ quy (dễ hiểu)

In [7]:
def x_value_segtree_rec(nums, k, queries):
    a = nums[:]
    n = len(a)
    tree = [None] * (4 * n)

    def build(node, l, r):
        if l == r:
            tree[node] = leaf(a[l], k); return
        m = (l + r) // 2
        build(2 * node, l, m); build(2 * node + 1, m + 1, r)
        tree[node] = combine(tree[2 * node], tree[2 * node + 1], k)

    def update(node, l, r, i):
        if l == r:
            tree[node] = leaf(a[i], k); return
        m = (l + r) // 2
        if i <= m: update(2 * node, l, m, i)
        else: update(2 * node + 1, m + 1, r, i)
        tree[node] = combine(tree[2 * node], tree[2 * node + 1], k)

    def query(node, l, r, ql):
        if r < ql:
            return IDENT(k)
        if l >= ql:
            return tree[node]
        m = (l + r) // 2
        return combine(query(2 * node, l, m, ql), query(2 * node + 1, m + 1, r, ql), k)

    build(1, 0, n - 1)
    res = []
    for idx, val, start, x in queries:
        a[idx] = val
        update(1, 0, n - 1, idx)
        res.append(query(1, 0, n - 1, start)[1][x])
    return res

for nums, k, qs, exp in examples:
    print(x_value_segtree_rec(nums, k, queries=qs), '| kỳ vọng', exp)

[2, 2, 2] | kỳ vọng [2, 2, 2]
[1, 0] | kỳ vọng [1, 0]
[5] | kỳ vọng [5]


### 7.3. Cách 4 — Cây phân đoạn lặp, lưu mảng phẳng (lời giải chính)

Lưu `P[node]` và `C[node]` (list độ dài `k`) trong hai mảng; xây từ lá lên; truy vấn hậu tố bằng cách gom hai phía như mục 3.4.

In [8]:
def x_value_segtree(nums, k, queries):
    n = len(nums)
    size = 1
    while size < n:
        size *= 2
    P = [1 % k] * (2 * size)
    C = [[0] * k for _ in range(2 * size)]
    for i, v in enumerate(nums):
        P[size + i] = v % k
        C[size + i][v % k] = 1

    def pull(i):
        l, r = 2 * i, 2 * i + 1
        pl = P[l]
        c = C[l][:]
        cr = C[r]
        for s in range(k):
            if cr[s]:
                c[pl * s % k] += cr[s]
        P[i] = pl * P[r] % k
        C[i] = c

    for i in range(size - 1, 0, -1):
        pull(i)

    res = []
    for idx, val, start, x in queries:
        i = size + idx
        P[i] = val % k
        C[i] = [0] * k
        C[i][val % k] = 1
        i >>= 1
        while i:
            pull(i)
            i >>= 1
        # truy vấn [start, n): gom trái (lp, lc) và phải (rp, rc)
        lo, hi = start + size, n + size
        lp, lc = 1 % k, [0] * k
        right_nodes = []
        while lo < hi:
            if lo & 1:
                c = lc[:]
                for s in range(k):
                    if C[lo][s]:
                        c[lp * s % k] += C[lo][s]
                lp, lc = lp * P[lo] % k, c
                lo += 1
            if hi & 1:
                hi -= 1
                right_nodes.append(hi)
            lo >>= 1
            hi >>= 1
        for node in reversed(right_nodes):
            c = lc[:]
            for s in range(k):
                if C[node][s]:
                    c[lp * s % k] += C[node][s]
            lp, lc = lp * P[node] % k, c
        res.append(lc[x])
    return res

for nums, k, qs, exp in examples:
    print(x_value_segtree(nums, k, qs), '| kỳ vọng', exp)

[2, 2, 2] | kỳ vọng [2, 2, 2]
[1, 0] | kỳ vọng [1, 0]
[5] | kỳ vọng [5]


Các nút bên phải được thu theo thứ tự **phải → trái** nên ta đảo lại trước khi gộp tiếp vào kết quả trái.

**Độ phức tạp:** xây `O(n·k)`, mỗi truy vấn `O(k log n)`.

## 8. TRACE từng bước

Trace cách 4 trên ví dụ 1: in lá và các nút được gộp cho từng truy vấn.

In [9]:
def trace(nums, k, queries):
    a = nums[:]
    for idx, val, start, x in queries:
        a[idx] = val
        seg = IDENT(k)
        steps = []
        for i in range(start, len(a)):
            seg = combine(seg, leaf(a[i], k), k)
            steps.append(f"thêm {a[i]}: P={seg[0]}, cnt={seg[1]}")
        print(f"truy vấn {[idx, val, start, x]}: mảng {a}, xét {a[start:]}")
        for s in steps:
            print('     ', s)
        print(f"  -> đáp án cnt[{x}] = {seg[1][x]}")

trace(*examples[0][:3])

truy vấn [2, 2, 0, 2]: mảng [1, 2, 2, 4, 5], xét [1, 2, 2, 4, 5]
      thêm 1: P=1, cnt=[0, 1, 0]
      thêm 2: P=2, cnt=[0, 1, 1]
      thêm 2: P=1, cnt=[0, 2, 1]
      thêm 4: P=1, cnt=[0, 3, 1]
      thêm 5: P=2, cnt=[0, 3, 2]
  -> đáp án cnt[2] = 2
truy vấn [3, 3, 3, 0]: mảng [1, 2, 2, 3, 5], xét [3, 5]
      thêm 3: P=0, cnt=[1, 0, 0]
      thêm 5: P=0, cnt=[2, 0, 0]
  -> đáp án cnt[0] = 2
truy vấn [0, 1, 0, 1]: mảng [1, 2, 2, 3, 5], xét [1, 2, 2, 3, 5]
      thêm 1: P=1, cnt=[0, 1, 0]
      thêm 2: P=2, cnt=[0, 1, 1]
      thêm 2: P=1, cnt=[0, 2, 1]
      thêm 3: P=0, cnt=[1, 2, 1]
      thêm 5: P=0, cnt=[2, 2, 1]
  -> đáp án cnt[1] = 2


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def resultArray(self, nums: List[int], k: int, queries: List[List[int]]) -> List[int]:
        return x_value_segtree(nums, k, queries)

for nums, k, qs, exp in examples:
    got = Solution().resultArray(nums, k, qs)
    print(got, 'OK' if got == exp else 'SAI')

[2, 2, 2] OK
[1, 0] OK
[5] OK


(Khi nộp, đặt `x_value_segtree` ngay trong file — LeetCode chấp nhận hàm phụ cấp module.)

## 10. Vì sao thuật toán đúng

1. **Đưa về tiền tố:** sau khi bỏ `nums[0..start−1]`, mỗi cách bỏ hậu tố (phần còn lại khác rỗng) tương ứng đúng một tiền tố khác rỗng của `nums[start..]`.
2. **Phép gộp đúng:** mọi tiền tố khác rỗng của `A·B` hoặc là tiền tố khác rỗng của `A`, hoặc là `A` nối với tiền tố khác rỗng của `B` (không trùng nhau). Tích của trường hợp sau ≡ `P(A)·(tích tiền tố của B)`.
3. **Kết hợp:** `combine(combine(A, B), C) = combine(A, combine(B, C))` (cả hai đếm tiền tố của `ABC`) ⇒ cây phân đoạn hợp lệ; thứ tự gộp trái→phải được giữ.
4. **Cập nhật:** chỉ các nút chứa `index` thay đổi; tính lại từ lá lên gốc.
5. **Đơn vị:** đoạn rỗng `(1 mod k, 0…0)` là phần tử trung hòa (với `k = 1`, `1 mod 1 = 0` vẫn đúng vì mọi số dư đều là 0).

## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 10^5`, `q ≤ 2·10^4`, `k ≤ 5`.

| Cách | Mỗi truy vấn | Tiền xử lý | Space |
|---|---|---|---|
| 1. Tính lại | `O(n)` | — | `O(n)` |
| 2. Chia khối | `O(k√n)` | `O(nk)` | `O(n + k√n)` |
| 3. Cây phân đoạn đệ quy | `O(k log n)` | `O(nk)` | `O(nk)` |
| 4. Cây phân đoạn lặp | `O(k log n)` | `O(nk)` | `O(nk)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {
    'blocks': x_value_blocks,
    'segtree_rec': x_value_segtree_rec,
    'segtree': x_value_segtree,
    'submit': lambda a, k, q: Solution().resultArray(a, k, q),
}
edge = [
    ([7], 1, [[0, 3, 0, 0]], [1]),                     # k = 1: mọi tiền tố dư 0
    ([5, 5, 5], 5, [[1, 1, 0, 0], [1, 1, 0, 1]], [3, 0]),   # [5,1,5]: mọi tích tiền tố ≡ 0 (mod 5)
    ([2, 3], 4, [[0, 2, 1, 3], [1, 2, 0, 0]], [1, 1]),
]
for nums, k, qs, exp in examples + edge:
    b = x_value_brute(nums, k, qs)
    assert b == exp, (nums, k, qs, b, exp)
    for name, f in impls.items():
        got = f(nums, k, qs)
        assert got == exp, (name, nums, k, qs, got, exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
random.seed(3525)

def rand_queries(n, q, k, vmax):
    return [[random.randrange(n), random.randint(1, vmax), random.randrange(n), random.randrange(k)] for _ in range(q)]

cnt = 0
for _ in range(400):
    n = random.randint(1, 30)
    k = random.randint(1, 5)
    nums = [random.randint(1, 12) for _ in range(n)]
    qs = rand_queries(n, random.randint(1, 20), k, 12)
    exp = x_value_brute(nums, k, qs)
    for name, f in impls.items():
        assert f(nums, k, qs) == exp, (name, nums, k, qs)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test ngẫu nhiên.')

Khớp vét cạn trên 400 bộ test ngẫu nhiên.


### 12.3. Ca lớn (`n = 10^5`, `q = 2·10^4`)

In [13]:
import time
random.seed(5)
n, q = 10**5, 2 * 10**4
for k in (1, 3, 5):
    nums = [random.randint(1, 10**9) for _ in range(n)]
    qs = rand_queries(n, q, k, 10**9)
    t = time.time(); a = Solution().resultArray(nums, k, qs); t1 = time.time() - t
    assert a[:20] == x_value_brute(nums, k, qs[:20])
    print(f"k={k}: cây phân đoạn lặp {t1:.2f}s")
nums = [random.randint(1, 10**9) for _ in range(n)]
qs = rand_queries(n, 2000, 5, 10**9)
t = time.time(); b1 = x_value_blocks(nums, 5, qs); tb = time.time() - t
t = time.time(); b2 = x_value_segtree(nums, 5, qs); ts = time.time() - t
assert b1 == b2
print(f"2.000 truy vấn: chia khối {tb:.2f}s | cây phân đoạn {ts:.2f}s")

k=1: cây phân đoạn lặp 0.46s


k=3: cây phân đoạn lặp 0.60s


k=5: cây phân đoạn lặp 0.68s


2.000 truy vấn: chia khối 0.67s | cây phân đoạn 0.22s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Gộp sai thứ tự (coi phép gộp giao hoán)

In [14]:
def combine_swapped(A, B, k):
    return combine(B, A, k)                     # SAI: đảo vai trò trái/phải

k = 3
seg_ok, seg_bad = IDENT(k), IDENT(k)
for v in [2, 1, 1]:
    seg_ok = combine(seg_ok, leaf(v, k), k)
    seg_bad = combine_swapped(seg_bad, leaf(v, k), k)
print('đúng (tiền tố của [2,1,1]):', seg_ok[1], '| sai:', seg_bad[1])

đúng (tiền tố của [2,1,1]): [0, 0, 3] | sai: [0, 2, 1]


**Vì sao sai:** tiền tố `[2]`, `[2,1]`, `[2,1,1]` đều dư 2 (mod 3); gộp ngược thứ tự sẽ đếm "tiền tố" của mảng đảo ngược (`[1]`, `[1,1]`, `[1,1,2]` → dư `1, 1, 2`).

### Lỗi 2 — Đếm cả tiền tố rỗng (tích = 1)

In [15]:
def brute_with_empty(nums, k, queries):
    res = x_value_brute(nums, k, queries)
    return [r + (1 % k == x) for r, (_, _, _, x) in zip(res, queries)]   # SAI: thêm tiền tố rỗng

case = examples[2]
print('sai :', brute_with_empty(*case[:3]), '| đúng:', case[3])

sai : [6] | đúng: [5]


**Vì sao sai:** phần còn lại phải **khác rỗng** (không được bỏ cả mảng).

### Lỗi 3 — Quên rằng cập nhật **được giữ lại**, còn việc bỏ tiền tố thì **không**

In [16]:
def wrong_persist_prefix(nums, k, queries):
    a, res = nums[:], []
    for idx, val, start, x in queries:
        a[idx] = val
        a = a[start:] if start < len(a) else a          # SAI: xóa thật tiền tố
        p, c = 1 % k, 0
        for v in a:
            p = p * v % k; c += p == x
        res.append(c)
    return res

case = examples[0]
print('sai :', wrong_persist_prefix(*case[:3]), '| đúng:', case[3])

sai : [2, 2, 1] | đúng: [2, 2, 2]


**Vì sao sai:** đề: "Only this step persists" — chỉ phép **gán** được giữ.

### Lỗi 4 — Tính tích thật rồi mới lấy modulo (tràn/ chậm)

In [17]:
import math
vals = [10**9] * 2000
t_big = math.prod(vals)
print('số bit của tích thật:', t_big.bit_length(), '-> lấy dư từng bước rẻ hơn nhiều:', math.prod(v % 5 for v in vals) % 5 == t_big % 5)

số bit của tích thật: 59795 -> lấy dư từng bước rẻ hơn nhiều: True


**Vì sao sai:** Python không tràn số nhưng tích `10^{18000}` rất tốn bộ nhớ/thời gian; ngôn ngữ khác thì tràn ngay. Luôn giữ số dư.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm tiền tố/ hậu tố thỏa điều kiện trên một đoạn, có cập nhật điểm" ⇒ cây phân đoạn với **thông tin tổng hợp** đủ để gộp.
- `k` nhỏ ⇒ lưu cả **phân bố** theo số dư (vector độ dài `k`).
- Phép gộp **không giao hoán** ⇒ truy vấn lặp phải gom hai phía riêng (hoặc dùng đệ quy).
- Bài cùng họ: "đếm đoạn con có tích ≡ x" (bản I — LC 3524) dùng DP theo số dư.

## 15. Biến thể của bài

1. Bản I (LC 3524): đếm mọi **đoạn con** (bỏ tiền tố và hậu tố tùy ý) có tích ≡ x ⇒ DP `dp[i][r]` số đoạn kết thúc tại `i`.
2. Thay tích bằng **tổng** mod `k` ⇒ cùng cấu trúc, `P` là tổng và phép gộp cộng.
3. Truy vấn đoạn `[l, r]` bất kỳ thay vì hậu tố ⇒ cây phân đoạn đã hỗ trợ sẵn.
4. `k` lớn (tới `10^5`) ⇒ vector quá lớn; cần ý tưởng khác (chẳng hạn chỉ hỏi một `x` và dùng nghịch đảo modulo khi `k` nguyên tố).

In [18]:
# Biến thể 1 (LC 3524): đếm mọi đoạn con theo số dư của tích
def count_subarrays_by_remainder(nums, k):
    total = [0] * k
    cur = [0] * k                       # số đoạn kết thúc tại vị trí hiện tại theo số dư
    for v in nums:
        nxt = [0] * k
        nxt[v % k] += 1
        for r in range(k):
            if cur[r]:
                nxt[r * v % k] += cur[r]
        cur = nxt
        for r in range(k):
            total[r] += cur[r]
    return total

nums = [1, 2, 3, 4, 5]
brute = [0] * 3
for i in range(5):
    p = 1
    for j in range(i, 5):
        p = p * nums[j] % 3
        brute[p] += 1
print(count_subarrays_by_remainder(nums, 3), brute)

[9, 2, 4] [9, 2, 4]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Khi nào dùng |
|---|---|---|---|
| 1 | Tính lại hậu tố | `O(n)` | Oracle |
| 2 | Chia khối | `O(k√n)` | Khi ngại cây phân đoạn |
| 3 | Cây phân đoạn đệ quy | `O(k log n)` | Dễ viết, dễ đúng |
| 4 | Cây phân đoạn lặp | `O(k log n)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `resultArray([3, 3, 3], 2, [[0, 3, 0, 1]])`.

**Bài 2 (tính tay):** `resultArray([2, 5, 1], 3, [[2, 2, 1, 1], [0, 1, 0, 2]])`.

**Bài 3 (điền TODO):** viết `segment_info(arr, k)` trả về `(P, cnt)` của một mảng.

**Bài 4 (tự viết lại):** biến thể 3 — truy vấn `[index, value, l, r, x]` trên đoạn `[l, r]`.

**Bài 5 (mini challenge):** biến thể 2 — dùng **tổng** mod `k` thay cho tích.

In [19]:
def segment_info_todo(arr, k):
    # TODO
    raise NotImplementedError

def range_query_todo(nums, k, queries):
    # TODO
    raise NotImplementedError

def sum_version_todo(nums, k, queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mọi tích là số lẻ. (2) Dư 1 khi chia 2. (3) `3`.

**Bài 2.** (1) Truy vấn 1: `[2,5,2]`, xét `[5,2]`: dư `2, 1` ⇒ `x = 1`: 1. (2) Truy vấn 2: `[1,5,2]`: dư `1, 2, 1` ⇒ `x = 2`: 1. (3) `[1, 1]`.

**Bài 3.** (1) Bắt đầu từ `IDENT(k)`. (2) Gộp từng lá. (3) So với vét cạn tiền tố.

**Bài 4.** (1) Truy vấn cây phân đoạn với `[l, r+1)`. (2) Giữ thứ tự gộp. (3) Vét cạn kiểm tra.

**Bài 5.** (1) `P` là tổng mod `k`. (2) Gộp: `c[(PA + s) % k] += cB[s]`. (3) Đơn vị `(0, 0…0)`.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
a1 = Solution().resultArray([3, 3, 3], 2, [[0, 3, 0, 1]])
a2 = Solution().resultArray([2, 5, 1], 3, [[2, 2, 1, 1], [0, 1, 0, 2]])
assert a1 == [3] == x_value_brute([3, 3, 3], 2, [[0, 3, 0, 1]])
assert a2 == [1, 1] == x_value_brute([2, 5, 1], 3, [[2, 2, 1, 1], [0, 1, 0, 2]])

def segment_info(arr, k):
    seg = IDENT(k)
    for v in arr:
        seg = combine(seg, leaf(v, k), k)
    return seg

random.seed(3)
for _ in range(300):
    k = random.randint(1, 5)
    arr = [random.randint(1, 20) for _ in range(random.randint(0, 10))]
    P, c = segment_info(arr, k)
    p, cc = 1 % k, [0] * k
    for v in arr:
        p = p * v % k; cc[p] += 1
    assert (P, c) == (p, cc)
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: đúng')

Bài 1: [3] | Bài 2: [1, 1] | Bài 3: đúng


In [21]:
# Bài 4: truy vấn đoạn [l, r]
def range_query(nums, k, queries):
    a = nums[:]
    n = len(a)
    size = 1
    while size < n:
        size *= 2
    T = [IDENT(k)] * (2 * size)
    for i, v in enumerate(a):
        T[size + i] = leaf(v, k)
    for i in range(size - 1, 0, -1):
        T[i] = combine(T[2 * i], T[2 * i + 1], k)
    res = []
    for idx, val, l, r, x in queries:
        i = size + idx
        T[i] = leaf(val, k)
        i >>= 1
        while i:
            T[i] = combine(T[2 * i], T[2 * i + 1], k); i >>= 1
        lo, hi = l + size, r + 1 + size
        left, right = IDENT(k), IDENT(k)
        while lo < hi:
            if lo & 1: left = combine(left, T[lo], k); lo += 1
            if hi & 1: hi -= 1; right = combine(T[hi], right, k)
            lo >>= 1; hi >>= 1
        res.append(combine(left, right, k)[1][x])
    return res

def range_brute(nums, k, queries):
    a, res = nums[:], []
    for idx, val, l, r, x in queries:
        a[idx] = val
        p, c = 1 % k, 0
        for v in a[l:r + 1]:
            p = p * v % k; c += p == x
        res.append(c)
    return res

random.seed(4)
for _ in range(200):
    n, k = random.randint(1, 20), random.randint(1, 5)
    nums = [random.randint(1, 15) for _ in range(n)]
    qs = []
    for _ in range(10):
        l = random.randrange(n)
        qs.append([random.randrange(n), random.randint(1, 15), l, random.randrange(l, n), random.randrange(k)])
    assert range_query(nums, k, qs) == range_brute(nums, k, qs)
print('Bài 4: khớp vét cạn')

Bài 4: khớp vét cạn


In [22]:
# Bài 5: phiên bản tổng
def sum_version(nums, k, queries):
    a, res = nums[:], []
    def info(arr):
        P, c = 0, [0] * k
        for v in arr:
            nc = [0] * k
            nc[(P + v) % k] = 1
            # gộp (P, c) với lá v: tiền tố cũ giữ nguyên, thêm tiền tố mới = toàn bộ + v
            c = [c[r] + nc[r] for r in range(k)]
            P = (P + v) % k
        return P, c
    for idx, val, start, x in queries:
        a[idx] = val
        res.append(info(a[start:])[1][x])
    return res

def sum_brute(nums, k, queries):
    a, res = nums[:], []
    for idx, val, start, x in queries:
        a[idx] = val
        s, c = 0, 0
        for v in a[start:]:
            s = (s + v) % k; c += s == x
        res.append(c)
    return res

random.seed(5)
for _ in range(200):
    n, k = random.randint(1, 15), random.randint(1, 5)
    nums = [random.randint(1, 15) for _ in range(n)]
    qs = rand_queries(n, 8, k, 15)
    assert sum_version(nums, k, qs) == sum_brute(nums, k, qs)
print('Bài 5: phiên bản tổng khớp vét cạn')

Bài 5: phiên bản tổng khớp vét cạn


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
lá v: (v % k, e[v % k]);   đơn vị: (1 % k, [0]*k)
combine((pa, ca), (pb, cb)): c = ca[:]; c[pa*s % k] += cb[s] với mọi s; P = pa*pb % k
cập nhật điểm: sửa lá, pull lên gốc
truy vấn [start, n): gom trái (theo thứ tự) + gom phải (đảo lại) -> cnt[x]
```

**Checklist:**

- [ ] Phép gộp đúng thứ tự (không giao hoán)?
- [ ] Không đếm tiền tố rỗng?
- [ ] Gán được giữ, bỏ tiền tố thì không?
- [ ] Mọi phép nhân đều lấy `% k`?
- [ ] Trường hợp `k = 1`?